In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
from skimage import io, transform
from tqdm import tqdm

# Imports Grounding DINO
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules

# Imports MedSAM
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference

# ==========================================
# FONCTIONS UTILITAIRES
# ==========================================
def calculate_dice(mask_true, mask_pred):
    m_true = np.asarray(mask_true).astype(bool)
    m_pred = np.asarray(mask_pred).astype(bool)
    if m_true.sum() + m_pred.sum() == 0:
        return 1.0
    intersection = np.logical_and(m_true, m_pred).sum()
    return 2 * intersection / (m_true.sum() + m_pred.sum())

# ==========================================
# 1. INITIALISATION DES MODÈLES
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Appareil détecté : {device}")

register_all_modules()

# --- MODIFICATION ICI : Chemins vers la config et le meilleur checkpoint Tumeur ---
dino_config = 'tumor_config.py'
dino_checkpoint = 'work_dirs/tumor_config/best_coco_bbox_mAP_epoch_7.pth'
dino_model = init_detector(dino_config, dino_checkpoint, device=device)

medsam_checkpoint = "MedSAM/work_dir/MedSAM/medsam_vit_b.pth"
medsam_model = sam_model_registry["vit_b"](checkpoint=medsam_checkpoint)
medsam_model = medsam_model.to(device)
medsam_model.eval()

# ==========================================
# 2. CONFIGURATION DES CHEMINS (MSD Pancreas)
# ==========================================
base_dir = "data/MSD_pancreas"
test_json_path = os.path.join(base_dir, "test.json")
output_folder = "data/outputs_medsam_dino_msd"

os.makedirs(output_folder, exist_ok=True)
os.makedirs("data/results", exist_ok=True)

# Chargement du fichier test.json
with open(test_json_path, 'r') as f:
    test_data = json.load(f)

test_images_list = test_data['images'] 

dice_list = []

print(f"\nDébut de l'évaluation sur les {len(test_images_list)} images du fichier test.json...")

# ==========================================
# 3. BOUCLE D'INFÉRENCE
# ==========================================
for img_info in tqdm(test_images_list, desc="Inférence Test Set"):
    # file_name contient "train/images/pancreas_XXX_sYY.png" ou "val/images/..."
    file_name = img_info['file_name']
    img_id = img_info['id']
    
    img_path = os.path.join(base_dir, file_name)
    
    # Déduction du chemin du masque (remplacement de 'images' par 'masks')
    mask_rel_path = file_name.replace("/images/", "/masks/")
    mask_path = os.path.join(base_dir, mask_rel_path)

    if not os.path.exists(img_path) or not os.path.exists(mask_path):
        continue
        
    img_np = io.imread(img_path)
    if len(img_np.shape) == 2:
        img_3c = np.repeat(img_np[:, :, None], 3, axis=-1)
    else:
        img_3c = img_np
    H, W, _ = img_3c.shape

    # --- MODIFICATION ICI : Isolement exclusif de la tumeur (label 2) ---
    true_seg_raw = io.imread(mask_path)
    true_seg = np.zeros_like(true_seg_raw, dtype=np.uint8)
    true_seg[true_seg_raw == 2] = 1

    # AJOUT : Vérification de la présence d'une tumeur dans la vérité terrain
    has_tumor = true_seg.sum() > 0

    full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

    with torch.no_grad():
        # --- MODIFICATION ICI : Le prompt devient "tumor." ---
        result = inference_detector(dino_model, img_path, text_prompt="tumor.")
        pred_instances = result.pred_instances
        
        scores = pred_instances.scores.cpu().numpy()
        bboxes = pred_instances.bboxes.cpu().numpy()
        
        mask_conf = scores > 0.1
        valid_boxes = bboxes[mask_conf]
        valid_scores = scores[mask_conf]

        if len(valid_boxes) > 0:
            best_idx = np.argmax(valid_scores)
            best_box = valid_boxes[best_idx]
            
            # Pre-processing MedSAM
            img_1024 = transform.resize(img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
            img_1024 = (img_1024 - img_1024.min()) / np.clip(img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None)
            img_1024_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
            
            image_embedding = medsam_model.image_encoder(img_1024_tensor)
            
            box_np = np.array([best_box]) 
            box_1024 = box_np / np.array([W, H, W, H]) * 1024
            
            medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)
            full_medsam_seg[medsam_seg > 0] = 1

    # Calcul et stockage
    dice_score = calculate_dice(true_seg, full_medsam_seg)
    dice_list.append({
        "image_id": img_id,
        "file_name": os.path.basename(file_name),
        "dice": dice_score,
        "has_tumor": has_tumor  # AJOUT : On stocke l'information sous forme de booléen
    })

# ==========================================
# 4. ANALYSE DES RÉSULTATS
# ==========================================
df = pd.DataFrame(dice_list)
df.to_csv("data/results/dice_final_report_msd_oneshot.csv", index=False)

# Séparation des données
df_tumor = df[df['has_tumor'] == True]
df_no_tumor = df[df['has_tumor'] == False]

print("\n" + "="*40)
print("RÉSULTATS GLOBAUX (Toutes les images)")
print("-" * 40)
print(f"DICE MOYEN : {df['dice'].mean():.4f}")
print(f"DICE MEDIAN : {df['dice'].median():.4f}")
print(f"MEILLEUR DICE : {df['dice'].max():.4f}")
print(f"PIRE DICE : {df['dice'].min():.4f}")

print("\n" + "="*40)
print(f"SCANS AVEC TUMEUR ({len(df_tumor)} images)")
print("-" * 40)
if not df_tumor.empty:
    print(f"DICE MOYEN : {df_tumor['dice'].mean():.4f}")
    print(f"DICE MEDIAN : {df_tumor['dice'].median():.4f}")
else:
    print("Aucune image avec tumeur dans ce set.")

print("\n" + "="*40)
print(f"SCANS SANS TUMEUR ({len(df_no_tumor)} images)")
print("-" * 40)
if not df_no_tumor.empty:
    print(f"DICE MOYEN : {df_no_tumor['dice'].mean():.4f}")
    print(f"DICE MEDIAN : {df_no_tumor['dice'].median():.4f}")
    print(f"FAUX POSITIFS (DICE = 0.0) : {len(df_no_tumor[df_no_tumor['dice'] == 0.0])} images")
else:
    print("Aucune image saine dans ce set.")
print("="*40)

/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


Appareil détecté : cuda


/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loads checkpoint by local backend from path: work_dirs/tumor_config/best_coco_bbox_mAP_epoch_7.pth


/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filena

...

noun_phrases: ['tumor']


Inférence Test Set:  99%|█████████▉| 99/100 [01:49<00:00,  1.05it/s][nltk_data] Downloading package punkt to ~/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmcv/cnn/bricks/transformer.py:524: UserWarning: position encoding of key ismissing in MultiheadAttention.
  warnings.warn(f'position encoding of key is'


noun_phrases: ['tumor']


Inférence Test Set: 100%|██████████| 100/100 [01:50<00:00,  1.11s/it]


RÉSULTATS GLOBAUX (Toutes les images)
----------------------------------------
DICE MOYEN : 0.3639
DICE MEDIAN : 0.1558
MEILLEUR DICE : 1.0000
PIRE DICE : 0.0000

SCANS AVEC TUMEUR (60 images)
----------------------------------------
DICE MOYEN : 0.4232
DICE MEDIAN : 0.5049

SCANS SANS TUMEUR (40 images)
----------------------------------------
DICE MOYEN : 0.2750
DICE MEDIAN : 0.0000
FAUX POSITIFS (DICE = 0.0) : 29 images
